# Get All Data Shortcuts

A `GraphModel` on a database connection loads only part of the model: creating a `FeederModel` reads the equipment in the container and the terminals and connectivity nodes that connect them, but not their attributes. For the objects of a class already in the graph, `network.get_all_edges(cim_class)` reads their attributes and associations and adds the objects they reference; `network.get_all_attributes(cim_class)` reads them without adding new objects.

The shortcuts in `cimgraph.utils` call `get_all_edges` for groups of classes:

| Function | Classes |
|---|---|
| `get_all_bus_data` | `ConnectivityNode`, `Terminal`, `TopologicalNode`, `TopologicalIsland` |
| `get_all_line_data` | `ACLineSegment`, `ACLineSegmentPhase`, `PerLengthPhaseImpedance`, `PhaseImpedanceData`, `WireSpacingInfo`, `WirePosition`, `OverheadWireInfo`, `ConcentricNeutralCableInfo`, `TapeShieldCableInfo` |
| `get_all_transformer_data` | `PowerTransformer`, `TransformerTank`, transformer ends, infos, impedances and tests, tap changers |
| `get_all_load_data` | `EnergySource`, `EnergyConsumer`, `ConformLoad`, `NonConformLoad`, `EnergyConsumerPhase`, `LoadResponseCharacteristic`, `PowerCutZone`, `House` |
| `get_all_inverter_data` | `PowerElectronicsConnection`, `PowerElectronicsConnectionPhase`, `BatteryUnit`, `PhotoVoltaicUnit`, `PowerElectronicsWindUnit` |
| `get_all_switch_data` | `Switch` and its subclasses, `SwitchPhase` |
| `get_all_limit_data` | `OperationalLimitSet`, `ActivePowerLimit`, `ApparentPowerLimit`, `VoltageLimit`, `CurrentLimit`, `OperationalLimitType` |
| `get_all_location_data` | `CoordinateSystem`, `Location`, `PositionPoint` |
| `get_all_measurement_data` | `Terminal`, `Measurement`, `Analog`, `Discrete`, `StringMeasurement` |
| `get_all_data` | all of the above, then every class found in the graph, until no new classes appear |

`XMLFile` and `JSONLDFile` read the whole file when the model is created, so these shortcuts are for database connections. The example below uses [`RDFlibConnection`](../03_databases/3_10_rdflib_parser.ipynb) on a local file, which needs no server (install it with `pip install cim-graph[rdflib]`).

## Load the Feeder

In [1]:
import os
os.environ['CIMG_CIM_PROFILE'] = 'cimhub_2023'
import cimprofiles.cim17.cimhub_2023 as cim
from cimgraph import utils
from cimgraph.databases import RDFlibConnection
from cimgraph.models import FeederModel

connection = RDFlibConnection(filename='../../tests/test_models/ieee13.xml')
feeder = cim.Feeder(mRID='49AD8E07-3BF9-A4E2-CB8F-C3722F837B62')
network = FeederModel(connection=connection, container=feeder, distributed=False)

def count_objects(network):
    return sum(len(objects) for objects in network.graph.values())

print(count_objects(network), 'objects of', len(network.graph), 'classes')

141 objects of 14 classes


The line 650632 is in the graph, but its attributes and associations are not loaded yet:

In [2]:
line = network.get_object('A04CDFB1-E951-4FC4-8882-0323CD70AE3C')
print(line.name, line.length, line.PerLengthImpedance)

None None None


## Line Data

In [3]:
utils.get_all_line_data(network)
print(count_objects(network), 'objects')
print(line.name, round(line.length.to('ft'), 1), 'ft', line.PerLengthImpedance.name)

212 objects
650632 2000.0 ft mtx601


## All Data

In [4]:
utils.get_all_data(network)
print(count_objects(network), 'objects of', len(network.graph), 'classes')

493 objects of 45 classes
